# 02 · Train with TorchDistributor — Single Node

단일 노드 토폴로지(1×1 / 1×N)를 한 노트북에서 비교합니다. multi-node(M×N)는 [`03-train_torch_distributor_MxN.ipynb`](03-train_torch_distributor_MxN.ipynb)에서 따로 다룹니다.

두 섹션의 호출 형태는 다음 표와 같습니다.

| 섹션 | 모드 | 호출 |
|------|------|------|
| 1×1  | DDP, world_size=1 | `TorchDistributor(num_processes=1, local_mode=True).run(train_fn_ddp, ...)` |
| 1×N  | DDP, single node  | `TorchDistributor(num_processes=N, local_mode=True).run(train_fn_ddp, ...)` |

두 섹션이 **같은 `train_fn_ddp`** 를 공유합니다. 모델·데이터·`CONFIG`가 모두 동일하고 차이는 `num_processes`뿐입니다. world_size=1에서는 DDP all_reduce가 사실상 no-op으로 동작하므로, 1×1을 standard PyTorch loop 대신 TorchDistributor로 돌려도 결과는 같고 **launcher·시그니처는 1×N·M×N과 일관**됩니다.

> `local_mode=True`는 **driver 노드 안에서** child 프로세스를 spawn한다는 뜻입니다 (single-node 클러스터의 driver=학습 노드). multi-node M×N에서는 `local_mode=False`로 worker 노드에 분산합니다. 상세는 [`concepts-distributed-training.md` — TorchDistributor `local_mode`](../00-foundations/concepts-distributed-training.md)를 참조합니다.

**클러스터 (이 노트북 전체 공용)**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0, Autoscaling off. 1×1 섹션은 GPU 1개만 사용해도 되므로 같은 클러스터에서 그대로 실행됩니다.

공통 모델·데이터 정의는 [`concepts-recommender-baseline.md`](../00-foundations/concepts-recommender-baseline.md), [`data-pipeline.md`](../00-foundations/data-pipeline.md)에, 패턴 함정은 [`debug-common-pitfalls.md`](../00-foundations/debug-common-pitfalls.md)에 정리되어 있습니다.

In [ ]:
%run ./00-setup

## 학습 함수 (1×1 / 1×N 공용)

TorchDistributor child가 실행할 entrypoint입니다. 1×1·1×N 섹션이 같은 함수를 사용하고, 호출 시 `num_processes`만 달라집니다. multi-node(M×N) 노트북과도 동일한 시그니처를 유지합니다.

worker 프로세스는 fresh Python으로 시작하므로 **함수 내부에 import·클래스 정의를 모두 둡니다**. closure로 driver 변수를 캡처하지 말고 모두 인자로 전달합니다 ([debug-common-pitfalls.md §2](../00-foundations/debug-common-pitfalls.md)). MLflow `run_id`는 driver에서 만든 뒤 인자로 넘겨, child 안에서 `mlflow.start_run(run_id=...)`로 attach합니다.

DDP early stopping은 다음과 같이 동작합니다. 매 epoch 끝에 모든 rank가 val_loss(local sum/count)를 `dist.all_reduce(SUM)`으로 합쳐 동일한 `avg_val_loss`를 얻고, 같은 `EarlyStopping.step` 결과로 동시에 break합니다. world_size=1에서도 `all_reduce`는 no-op으로 통과하므로 코드 분기가 필요 없습니다.

In [ ]:
def train_fn_ddp(
    run_id,
    db_host,
    db_token,
    data_dir,
    ckpt_path,
    n_users,
    n_items,
    emb_dim,
    tower_hidden,
    batch_size,
    num_epochs,
    max_steps_per_epoch,
    patience,
    min_delta,
    topology,
):
    """TorchDistributor entrypoint. 1×1(num_processes=1) / 1×N에서 같은 함수를 사용한다.

    `data_dir`은 train/val 서브디렉토리를 포함한다. epoch 단위 루프 + DDP val all_reduce + EarlyStopping.
    """
    import os

    import mlflow
    import pyarrow.parquet as pq
    import torch
    import torch.distributed as dist
    import torch.nn as nn
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader, TensorDataset
    from torch.utils.data.distributed import DistributedSampler

    os.environ["DATABRICKS_HOST"] = db_host
    os.environ["DATABRICKS_TOKEN"] = db_token

    dist.init_process_group("nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    global_rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    class TwoTowerMLP(nn.Module):
        def __init__(self, n_users, n_items, emb_dim, tower_hidden):
            super().__init__()
            self.user_emb = nn.Embedding(n_users, emb_dim)
            self.item_emb = nn.Embedding(n_items, emb_dim)
            self.user_tower = self._make_tower(emb_dim, tower_hidden)
            self.item_tower = self._make_tower(emb_dim, tower_hidden)

        @staticmethod
        def _make_tower(in_dim, hidden):
            layers, prev = [], in_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.ReLU()]
                prev = h
            return nn.Sequential(*layers)

        def forward(self, user_ids, item_ids):
            u = self.user_tower(self.user_emb(user_ids))
            i = self.item_tower(self.item_emb(item_ids))
            return (u * i).sum(dim=-1)

    class EarlyStopping:
        def __init__(self, patience, min_delta):
            self.patience = patience
            self.min_delta = min_delta
            self.best = float("inf")
            self.counter = 0

        def step(self, val_loss):
            if val_loss < self.best - self.min_delta:
                self.best = val_loss
                self.counter = 0
                return False
            self.counter += 1
            return self.counter >= self.patience

    def load_split(split):
        split_dir = os.path.join(data_dir, split)
        files = sorted(
            os.path.join(split_dir, f) for f in os.listdir(split_dir) if f.endswith(".parquet")
        )
        table = pq.read_table(files, columns=["user_id", "item_id", "label"])
        return TensorDataset(
            torch.from_numpy(table.column("user_id").to_numpy()),
            torch.from_numpy(table.column("item_id").to_numpy()),
            torch.from_numpy(table.column("label").to_numpy()),
        )

    train_dataset = load_split("train")
    val_dataset = load_split("val")
    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=global_rank, drop_last=True)
    val_sampler = DistributedSampler(val_dataset, num_replicas=world_size, rank=global_rank, shuffle=False, drop_last=False)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, sampler=val_sampler, num_workers=2, pin_memory=True)

    model = TwoTowerMLP(n_users, n_items, emb_dim, tower_hidden).to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    early_stop = EarlyStopping(patience=patience, min_delta=min_delta)

    if global_rank == 0:
        mlflow.start_run(run_id=run_id, log_system_metrics=True)
        mlflow.log_params({
            "topology": topology,
            "world_size": world_size,
            "n_users": n_users,
            "n_items": n_items,
            "emb_dim": emb_dim,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "max_steps_per_epoch": max_steps_per_epoch,
            "patience": patience,
            "min_delta": min_delta,
        })

    global_step = 0
    for epoch in range(num_epochs):
        train_sampler.set_epoch(epoch)
        model.train()
        for step, (u, i, y) in enumerate(train_loader):
            if step >= max_steps_per_epoch:
                break
            u, i, y = u.to(device), i.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(u, i)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            if global_rank == 0 and step % 20 == 0:
                mlflow.log_metric("train/loss", loss.item(), step=global_step)
            global_step += 1

        model.eval()
        local_loss_sum = torch.zeros(1, device=device)
        local_n = torch.zeros(1, device=device)
        with torch.no_grad():
            for u, i, y in val_loader:
                u, i, y = u.to(device), i.to(device), y.to(device)
                logits = model(u, i)
                loss = loss_fn(logits, y)
                local_loss_sum += loss * y.size(0)
                local_n += y.size(0)
        dist.all_reduce(local_loss_sum, op=dist.ReduceOp.SUM)
        dist.all_reduce(local_n, op=dist.ReduceOp.SUM)
        val_loss = (local_loss_sum / local_n).item()

        if global_rank == 0:
            mlflow.log_metric("val/loss", val_loss, step=epoch)
            print(f"[rank=0] epoch={epoch:2d} val_loss={val_loss:.6f} (best={early_stop.best:.6f} counter={early_stop.counter})")

        if early_stop.step(val_loss):
            if global_rank == 0:
                print(f"early stop at epoch {epoch} (best val_loss={early_stop.best:.6f})")
                mlflow.log_metric("early_stop_epoch", epoch)
            break

    dist.barrier()
    if global_rank == 0:
        torch.save({"model": model.module.state_dict()}, ckpt_path)
        mlflow.log_param("ckpt_path", ckpt_path)
        mlflow.end_run()
        print(f"saved {ckpt_path}")
    dist.destroy_process_group()
    return "ok"

## 1×1 GPU

`TorchDistributor(num_processes=1, local_mode=True, use_gpu=True).run(train_fn_ddp, ...)`로 호출합니다. **launcher·학습 함수는 1×N·M×N과 완전히 동일**하고 `num_processes`만 1입니다. world_size=1이면 DDP의 all_reduce는 사실상 no-op이라 1-GPU 학습과 결과가 같습니다.

이 패턴은 분산 학습 코드를 처음 작성할 때 1×1으로 먼저 검증한 뒤 `num_processes`만 키워 1×N → M×N으로 확장할 수 있다는 이점이 있습니다. driver 노트북 셀에 별도의 standard PyTorch loop를 두지 않아 같은 코드가 모든 토폴로지에서 재사용됩니다.

In [ ]:
import os

import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="1x1", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=1,
    local_mode=True,
    use_gpu=True,
).run(
    train_fn_ddp,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "1x1.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="1x1",
)

## 1×N GPU

단일 노드, N GPU 구성입니다. 위에서 정의한 같은 `train_fn_ddp`를 그대로 사용하고 `num_processes`만 N으로 바꿉니다. global batch는 `batch_size_per_gpu × N`으로 커집니다.

In [ ]:
from pyspark.ml.torch.distributor import TorchDistributor

NUM_GPUS_PER_NODE = 4  # 클러스터 driver의 GPU 수에 맞춰 조정

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="1xN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_GPUS_PER_NODE,
    local_mode=True,
    use_gpu=True,
).run(
    train_fn_ddp,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "1xN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="1xN",
)